<div dir="rtl">

# الدرس 42: الشبكات التكرارية وحدودها

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### المثال المحلول، بيدنا ثم بباي تورش

</div>

In [ ]:
import math
import torch
from torch import nn

x = [1.0, 0.0, 0.0]
h = 0.0
for t, xt in enumerate(x, start=1):
    h = math.tanh(0.5 * xt + 0.8 * h)
    print(f"h{t} = {h:.4f}")

rnn = nn.RNN(input_size=1, hidden_size=1, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.fill_(0.5)                # W_x
    rnn.weight_hh_l0.fill_(0.8)                # W_h
    rnn.bias_ih_l0.zero_(); rnn.bias_hh_l0.zero_()
seq = torch.tensor(x).reshape(1, 3, 1)         # (batch, time, features)
outputs, h_last = rnn(seq)
print("PyTorch:", outputs.detach().flatten().numpy().round(4))

<div dir="rtl">

### التدرّج عبر الزمن

</div>

In [ ]:
for T in [1, 3, 5, 10, 20, 50]:
    xs = torch.zeros(T)
    xs[0] = 1.0
    xs.requires_grad_(True)
    h = torch.tensor(0.0)
    for t in range(T):
        h = torch.tanh(0.5 * xs[t] + 0.8 * h)
    h.backward()
    print(f"T = {T:2}: d h_T / d x_1 = {xs.grad[0].item():.1e}")

<div dir="rtl">

### تجربة: هل تتذكر أول رقم؟

</div>

In [ ]:
def make_batch(n, T, g):
    first = torch.randint(0, 2, (n,), generator=g)          # the bit to remember
    x = torch.randn(n, T, 1, generator=g) * 0.2             # small noise everywhere
    x[:, 0, 0] = first.float() * 2 - 1                      # +1 or -1 at the first step
    return x, first

class Reader(nn.Module):
    def __init__(self, kind, hidden=32):
        super().__init__()
        self.rnn = (nn.RNN if kind == "RNN" else nn.LSTM)(1, hidden, batch_first=True)
        if kind == "LSTM":                      # start with the forget gate open: remember by default
            with torch.no_grad():
                for name, p in self.rnn.named_parameters():
                    if "bias" in name:
                        p[hidden:2 * hidden].fill_(1.5)   # two bias vectors: 1.5 + 1.5 = 3
        self.head = nn.Linear(hidden, 2)

    def forward(self, x):
        outputs, _ = self.rnn(x)
        return self.head(outputs[:, -1])        # decide from the last hidden state

def run(kind, T, steps=500, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    net = Reader(kind)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        x, y = make_batch(64, T, g)
        loss = loss_fn(net(x), y)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)   # against exploding gradients
        opt.step()
    x, y = make_batch(1000, T, torch.Generator().manual_seed(123))
    with torch.no_grad():
        return (net(x).argmax(1) == y).float().mean().item()

for T in [10, 30, 60]:
    print(f"T = {T:3}: RNN accuracy {run('RNN', T):.3f} | LSTM accuracy {run('LSTM', T):.3f}")

<div dir="rtl">

## تمرين: أطول من ذلك

كرّر تجربة التذكر بطول 100 خطوة، بثلاث بذور عشوائية مختلفة لكل شبكة: 0، و 1، و 2.

1. ما دقة كل شبكة، لكل بذرة؟
2. ماذا تستنتج عن الذاكرة الطويلة مع تسلسلات أطول بكثير؟

</div>

In [ ]:
# اكتب حلّك هنا
